# 03 — Tiny GPT v3: load & chat (no retraining)

Notebook **02** *builds and trains* the from-scratch GPT (~30 min). This notebook is the **consumer**: it **loads the saved checkpoint in ~1 second** and lets you generate / chat as much as you want — no training, ever.

**Prerequisite — mint the checkpoint once.** If you've never saved one, run this in a terminal (it's the headless twin of 02's training cell):

```bash
uv run python notebooks/train_v2_checkpoint.py        # ~30 min, one time
```

That writes `notebooks/checkpoints/tiny_gpt_v2/` (weights + tokenizer + config). After that, just re-run this notebook whenever you want to play.

> **Reality check:** this is a *TinyStories* model, not an instruction-tuned assistant. It doesn't answer questions — it *continues* text. Feed it a story opener ("Once upon a time…", "The dragon looked at the boy and said…") rather than "What is 2+2?".

In [1]:
import time
import tiny_gpt   # local module (notebooks/tiny_gpt.py) — the model class + load/generate

t0 = time.time()
model, tok, cfg = tiny_gpt.load("checkpoints/tiny_gpt_v2")
print(f"loaded in {time.time()-t0:.2f}s  |  config = {vars(cfg)}")

loaded in 0.06s  |  config = {'block_size': 256, 'n_embd': 384, 'n_head': 6, 'n_layer': 6, 'vocab_size': 8192, 'eos_token': '<|endstory|>'}


## Generation playground

`tiny_gpt.generate(model, tok, cfg, prompt, n_new=..., temperature=...)` returns a full completion. Re-run this cell with different prompts as often as you like — it reuses the model already in memory.

In [2]:
for prompt in ["Once upon a time", "The dragon looked at the boy and said", "In the dark forest"]:
    print("="*70)
    print(tiny_gpt.generate(model, tok, cfg, prompt, n_new=150, temperature=0.8))
    print()

Once upon a time, there was a little girl named Lily. She loved to draw with her colorful pastel colors. She would draw with a big, beautiful sun and a happy heart.

One day, Lily's friend came over to play. Lily said, "Hi scary cat, can you show me the pastel colors?" Her friend said, "Yes, I want to love to draw on the toy." Lily smiled and said, "Thank you, Lily!"

The two of them continued to play and have fun together. But, Lily accidentally dropped the new crayon on her toy doll. She was sad because she couldn't play with it. Her friend helped her look for it. They went home and Lily's house was fixed, and all the things

The dragon looked at the boy and said, "I can help you, little boy." The bird was happy and said, "Thank you, little boy!" And they continued to play in the park.



In the dark forest, there was a big tree. The tree had a tree that would hide under the tree. The pine tree was a big hole in the tree.

One sunny day, a little squirrel got stuck in the hole. The squirrel had a big bone. The squirrel wanted to find a bone.

The squirrel went inside and found a big hole. The squirrel let go and walked inside. The squirrel was safe and happy to be free. The squirrel was now safe with the shiny nut. They were both happy friends, and they played together all day.



## Temperature sweep — the one knob worth feeling

Same prompt, rising temperature. Low (~0.4) is coherent but repetitive; high (~1.2) is creative but loopier. (Recall from the handoff: temperature only *bites* on a confident model — v2 is trained enough to show the spread.)

In [3]:
for temp in (0.4, 0.7, 1.0, 1.2):
    print(f"--- temperature {temp} ---")
    print(tiny_gpt.generate(model, tok, cfg, "Once upon a time", n_new=120, temperature=temp))
    print()

--- temperature 0.4 ---
Once upon a time, there was a little girl named Lily. She loved to play with her toys and her friends. One day, Lily's mom asked her to clean up her toys and put them away. Lily didn't want to clean up, but she knew she needed to be clean.

So, Lily's mom helped her clean up the mess and they made a big mess. Lily was happy that her mom helped her clean up. From that day on, Lily always made sure to clean up her toys so she could play with her toys again.

--- temperature 0.7 ---


Once upon a time there were two friends who loved to play together. One day they decided to make a menu with lots of carrots.

They used soap to make food, salad and salad. It was the best soup they keep them all. It was a delicious surprise, they both couldn't wait!

As they were mixing the food, they found a very special treat inside. They wanted the cheese! So they stepped into the salad and it was delicious!

They both enjoyed the delicious cheese and the family ate it together. They were so happy and couldn't wait to take it out and take

--- temperature 1.0 ---
Once upon a time, there was a little girl named Lily. She lived in a big house with her mommy, daddy, and even a cat. One day, mommy asked her to go to the grocery store. Lily was so excited to go!

At the store, Lily saw a big toy that fit outside her out of the store. She asked her mommy, "Mommy, look! I want this one!" Her mommy replied, "Well, if it likes to go away from other things. You must put it in a bag so he doe

Once upon a time there was a kind and adorable veterinarian. Every day she would dance in the pool, twirl in the garden, feeling safe.

One day, in the pet mouse sniffed up and snuggled himself for against his face. As he blew out his paw in the grass, soon the postman went out and slept.

From then on, the dog would read his number and he would say "cusbanddies. Pretty soon you accomief, I don't suffer or grow up even faster." In the end, the dry page was closer to anything wagged and ask the elderly judge smiled. In



## Interactive chat

Run the cell below and type prompts at the box that appears. Tokens stream in as they're generated. Type `/quit` to stop, `/temp 0.6` or `/tokens 150` to adjust on the fly.

*(Prefer a real terminal? `uv run python notebooks/chat.py` gives the same REPL outside Jupyter.)*

In [ ]:
temp, ntok = 0.8, 200
print("Type a prompt (/temp N, /tokens N, /quit to stop).")
while True:
    prompt = input("\nyou \u25b8 ").strip()
    if not prompt:
        continue
    if prompt in ("/quit", "/exit", "/q"):
        break
    if prompt.startswith("/temp"):
        temp = float(prompt.split()[1]); print(f"  temperature = {temp}"); continue
    if prompt.startswith("/tokens"):
        ntok = int(prompt.split()[1]); print(f"  tokens = {ntok}"); continue
    print("gpt \u25b8 ", end="", flush=True)
    for delta in tiny_gpt.stream(model, tok, cfg, prompt, n_new=ntok, temperature=temp):
        print(delta, end="", flush=True)
    print()